# PaySim Fraud Detection Model Comparison

This notebook compares five classifiers across pre-authorization and post-authorization feature regimes using the same stratified train, validation, and test partitions. It reuses the project pipeline and feature definitions from `train.py` and `features.py`.

The pre-authorization model uses only fields available before approval and remains the deployable model for live screening. Post-authorization results are retrospective only: completed-transaction balances and balance-mismatch features are unavailable in real time and can cause temporal data leakage.

Models are selected by validation PR-AUC. If Random Forest leads XGBoost by no more than 0.01 PR-AUC and XGBoost trains faster, XGBoost is preferred. The held-out test set is used only to report final metrics.


## 1. Load PaySim data

Set `SAMPLE_ROWS` to `0` to use the full CSV. Sampling is reproducible. The fraud label is kept separate from the input features.


In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

from features import PRE_AUTH_RAW_COLS, POST_AUTH_RAW_COLS
from train import SEED, MODELS_DIR, XGB_RF_SELECTION_TOLERANCE, run_regime

DATA_PATH = Path("data/PS_20174392719_1491204439457_log.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../PS_20174392719_1491204439457_log.csv")
SAMPLE_ROWS = 1_000_000  # 0 uses all available rows

raw_cols = list(dict.fromkeys(PRE_AUTH_RAW_COLS + POST_AUTH_RAW_COLS))
assert "isFraud" not in raw_cols, "Target leakage: isFraud must not be a feature"
df = pd.read_csv(DATA_PATH, usecols=raw_cols + ["isFraud"])
if SAMPLE_ROWS and len(df) > SAMPLE_ROWS:
    df = df.sample(SAMPLE_ROWS, random_state=SEED)

X = df[raw_cols]
y = df["isFraud"].astype(int)
assert "isFraud" not in X.columns
print(f"Rows: {len(df):,}")
print(f"Fraud transactions: {int(y.sum()):,} ({y.mean():.3%})")
df.head()


Rows: 1,000,000
Fraud transactions: 1,266 (0.127%)


## 2. Create shared data partitions

The same stratified partitions are used for both feature sets, so the comparison reflects feature availability rather than different transactions. The validation set is used to select a model. The test set remains untouched until final reporting.


In [2]:
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=0.50, stratify=y_hold, random_state=SEED
)

print(f"Train:      {len(X_train):,}")
print(f"Validation: {len(X_val):,}")
print(f"Test:       {len(X_test):,}")


Train:      700,000
Validation: 150,000
Test:       150,000


## 3. Define the feature availability regimes

**Pre-authorization** includes transaction time, type, amount, and sender/receiver balances before the transaction. These are the fields available to the deployed screening API.

**Post-authorization** adds sender and receiver balances after the transaction, plus balance discrepancy features derived from those values. This can help quantify retrospective signal, but it must not be interpreted as live pre-authorization performance.


In [3]:
print("Pre-authorization input fields:")
print(PRE_AUTH_RAW_COLS)
print("\nPost-authorization input fields:")
print(POST_AUTH_RAW_COLS)


Pre-authorization input fields:
['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']

Post-authorization input fields:
['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'newbalanceOrig', 'newbalanceDest']


## 4. Train and compare the candidate models

`run_regime` fits the same five candidate models with the project's existing preprocessing, class weights, and random seed. Each regime gets its own comparison table sorted by validation PR-AUC. Normally the highest validation PR-AUC wins; when Random Forest leads XGBoost by no more than 0.01 and XGBoost trained faster, XGBoost is preferred. The table's other metrics are measured on the held-out test set at threshold 0.5. The test set does not select a model.


In [4]:
def show_regime_comparison(title, regime_results):
    rows = []
    for model_name, test_metrics in regime_results["test"].items():
        validation_metrics = regime_results["validation"][model_name]
        rows.append({
            "Model": model_name,
            "Validation PR-AUC": validation_metrics["pr_auc"],
            "PR-AUC": test_metrics["pr_auc"],
            "ROC-AUC": test_metrics["roc_auc"],
            "Precision @ 0.5": test_metrics["precision"],
            "Recall @ 0.5": test_metrics["recall"],
            "F1-score @ 0.5": test_metrics["f1"],
            "False Negatives": test_metrics["fn"],
            "False Positives": test_metrics["fp"],
            "Training Time (seconds)": test_metrics["train_seconds"],
        })
    table = pd.DataFrame(rows).sort_values(
        "Validation PR-AUC", ascending=False
    ).reset_index(drop=True)
    print(title)
    display(table.style
        .format({
            "Validation PR-AUC": "{:.4f}",
            "PR-AUC": "{:.4f}",
            "ROC-AUC": "{:.4f}",
            "Precision @ 0.5": "{:.4f}",
            "Recall @ 0.5": "{:.4f}",
            "F1-score @ 0.5": "{:.4f}",
            "Training Time (seconds)": "{:.2f}",
        })
        .highlight_max(subset=["Validation PR-AUC"], color="#b7e4c7"))
    print(f"Selected model: {regime_results['best_model']}")
    print(f"Selection rule: {regime_results['selection_rule']}")


pre_results, pre_model = run_regime(
    "pre_authorization", PRE_AUTH_RAW_COLS,
    X_train, y_train, X_val, y_val, X_test, y_test,
)
show_regime_comparison("Pre-Authorization Model Comparison", pre_results)


pre_authorization  Logistic Regression  validation PR-AUC=0.1609
pre_authorization  Decision Tree        validation PR-AUC=0.6011
pre_authorization  Random Forest        validation PR-AUC=0.6684
pre_authorization  Extra Trees          validation PR-AUC=0.6269
pre_authorization  XGBoost              validation PR-AUC=0.8355
Pre-Authorization Model Comparison
Selected model: XGBoost
Selection rule: highest validation PR-AUC


,Model,Validation PR-AUC,PR-AUC,ROC-AUC,Precision @ 0.5,Recall @ 0.5,F1-score @ 0.5,False Negatives,False Positives,Training Time (seconds)
0,XGBoost,0.8355,0.8472,0.9992,0.2543,0.9316,0.3995,13,519,1.30
1,Random Forest,0.6684,0.7491,0.9958,0.1665,0.8368,0.2777,31,796,8.51
2,Extra Trees,0.6269,0.6452,0.9820,0.0224,0.8684,0.0437,25,7193,3.61
3,Decision Tree,0.6011,0.5911,0.9444,0.0474,0.8947,0.0900,20,3416,1.01
4,Logistic Regression,0.1609,0.1612,0.9729,0.0152,0.8895,0.0299,21,10930,1.38


The pre-authorization table compares all five models using fields available when a payment is screened. The row with the highest validation PR-AUC is highlighted. If Random Forest leads XGBoost by no more than 0.01 PR-AUC and XGBoost trained faster, the selected model is XGBoost; the selection is printed below the table. The remaining performance columns are held-out test metrics.


In [5]:
post_results, _ = run_regime(
    "post_authorization", POST_AUTH_RAW_COLS,
    X_train, y_train, X_val, y_val, X_test, y_test,
)
show_regime_comparison(
    "Post-Authorization Model Comparison (Retrospective Only)", post_results
)


post_authorization Logistic Regression  validation PR-AUC=0.5631
post_authorization Decision Tree        validation PR-AUC=1.0000
post_authorization Random Forest        validation PR-AUC=1.0000
post_authorization Extra Trees          validation PR-AUC=0.8093
post_authorization XGBoost              validation PR-AUC=0.9987
Post-Authorization Model Comparison (Retrospective Only)
Selected model: Decision Tree
Selection rule: highest validation PR-AUC


,Model,Validation PR-AUC,PR-AUC,ROC-AUC,Precision @ 0.5,Recall @ 0.5,F1-score @ 0.5,False Negatives,False Positives,Training Time (seconds)
0,Decision Tree,1.0000,1.0000,1.0000,0.9896,1.0000,0.9948,0,2,1.05
1,Random Forest,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0,0,8.29
2,XGBoost,0.9987,0.9984,1.0000,0.9896,1.0000,0.9948,0,2,1.33
3,Extra Trees,0.8093,0.8210,0.9963,0.0385,0.9632,0.0741,7,4569,4.57
4,Logistic Regression,0.5631,0.5943,0.9912,0.0261,0.9368,0.0507,12,6651,0.65


The post-authorization table uses completed-transaction balances and derived balance-mismatch features, evaluated on the same train, validation, and test rows as the pre-authorization experiment. Its highlighted row has the highest validation PR-AUC.

**Warning:** post-authorization features are unavailable for real-time fraud screening and can cause temporal data leakage. These retrospective results must not be interpreted as live fraud-detection performance.

## 6. Save the pre-authorization model and comparison metrics

Only the selected pre-authorization model is saved to `models/best_model.joblib` for deployment. `models/metrics.json` records both regimes separately, including each regime's selected model and selection rule.


In [6]:
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(pre_model, MODELS_DIR / "best_model.joblib")

metrics = {
    "protocol": "stratified 70/15/15; random_state=42; fixed threshold=0.5",
    "best_model": pre_results["best_model"],
    "active_feature_regime": "pre_authorization",
    "rows_used": int(len(df)),
    "fraud_rate": float(y.mean()),
    "split_rows": {
        "train": len(X_train), "validation": len(X_val), "test": len(X_test),
    },
    "results": pre_results["test"],
    "regimes": {
        "pre_authorization": pre_results,
        "post_authorization": post_results,
    },
    "selection_metric": "validation PR-AUC (average precision)",
    "selection_policy": (
        "Choose the highest validation PR-AUC; if Random Forest leads XGBoost "
        f"by no more than {XGB_RF_SELECTION_TOLERANCE:.2f} and XGBoost trains faster, prefer XGBoost."
    ),
    "test_set_used_for_selection": False,
}
(MODELS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("Selected pre-authorization model for deployment:", pre_results["best_model"])
print("Selected post-authorization model (retrospective):", post_results["best_model"])
print("Saved models/best_model.joblib and models/metrics.json")


Selected pre-authorization model for deployment: XGBoost
Selected post-authorization model (retrospective): Decision Tree
Saved models/best_model.joblib and models/metrics.json


## How to interpret the comparison

Use the pre-authorization table to describe expected screening performance because it uses only fields available at decision time. The post-authorization table is a retrospective comparison; post-transaction balances and mismatch features are not available for live screening and may leak information from after the decision. PaySim is simulated data, so results may not match performance on real transactions.
